**1. Set up**

This notebook is where we evaluate the models. We are loading the models, running them on the test set, measuring how well they perform, and finding the tweets where they disagree. Those disagreement cases get saved for when we do the XAI analysis. We start by mounting Drive, importing everything we need, and confirming all 4 model paths are accessible.

In [ ]:

# mount Google Drive and import all libraries needed

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, classification_report
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


PROJECT_PATH = '/content/drive/Shareddrives/Cos760'

# label mappings
LBL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LBL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LANGUAGES = ['hausa', 'kinyarwanda']

# model paths
MODEL_PATHS = {
    'xlmr': {
        lang: os.path.join(PROJECT_PATH, f'models/xlmr/{lang}/final')
        for lang in LANGUAGES
    },
    'afriberta': {
        lang: os.path.join(PROJECT_PATH, f'models/afriberta/{lang}/final')
        for lang in LANGUAGES
    }
}

print("Setup complete. Model paths:")
for model_name, langs in MODEL_PATHS.items():
    for lang, path in langs.items():
        exists = os.path.exists(path)
        print(f"  [{('OK' if exists else 'MISSING')}] {model_name}/{lang}: {path}")

Mounted at /content/drive
Using device: cuda
Setup complete. Model paths:
  [OK] xlmr/hausa: /content/drive/Shareddrives/Cos760/models/xlmr/hausa/final
  [OK] xlmr/kinyarwanda: /content/drive/Shareddrives/Cos760/models/xlmr/kinyarwanda/final
  [OK] afriberta/hausa: /content/drive/Shareddrives/Cos760/models/afriberta/hausa/final
  [OK] afriberta/kinyarwanda: /content/drive/Shareddrives/Cos760/models/afriberta/kinyarwanda/final


**2. Load test data**

In this cell we load the test splits for both Hausa and Kinyarwanda from the cleaned CSVs produced in notebook 01. We then print the label distribution for each language so we have a clear picture of class balance going into evaluation.

In [ ]:

#loading test data
test_dfs = {}

for lang in LANGUAGES:
    path = os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_test_cleaned.csv')
    df = pd.read_csv(path)
    test_dfs[lang] = df
    print(f"\n{lang.upper()} test set: {len(df)} samples")
    print(f"  Label distribution:")
    print(df['label'].value_counts().to_string())


HAUSA test set: 5303 samples
  Label distribution:
label
neutral     1789
negative    1759
positive    1755

KINYARWANDA test set: 1026 samples
  Label distribution:
label
neutral     393
negative    355
positive    278


**3. Dataset class and inference function**

Here we define the two building blocks needed to run inference. The SentimentDataset class (from notebook 02) wraps our dataframe into a format PyTorch can read batch by batch. The run_inference() function takes a saved model path and a test dataframe, loads the model, and returns a predicted label for every individual tweet. Capturing per-tweet predictions is what allows us to do the disagreement analysis later.


In [ ]:
#Dataset Class
class SentimentDataset(Dataset): #from notebook 02
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tweet = str(self.data.iloc[idx]['cleaned_tweet'])
        encoding = self.tokenizer(
            tweet,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }

 # run_inference() loads a saved model, feeds it every tweet in the test set, and returns a predicted sentiment label for each  one.
def run_inference(model_path, test_df, batch_size=32):

    # load the tokenizer and model from the saved directory
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device)

    # set model to evaluation mode
    model.eval()

    # wrap the test dataframe in our Dataset class
    dataset = SentimentDataset(test_df, tokenizer)

    # DataLoader handles batching automatically
    loader = DataLoader(dataset, batch_size=batch_size)

    all_predictions = []

    # torch.no_grad() disables gradient tracking since we're just predicting, not learning
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask) # get raw scores (logits) for each class

            # pick the class with the highest score as the prediction
            predictions = torch.argmax(outputs.logits, dim=-1)
            all_predictions.extend(predictions.cpu().numpy())

    return all_predictions

**4. Running inference**

Now we call run_inference() for all 4 model and language combinations, i.e. XLM-R and AfriBERTa on both Hausa and Kinyarwanda. The predictions are stored in a dictionary structure so we can retrieve any combination cleanly in the cells that follow.

In [ ]:
# we call run_inference() for each model + language combo and store the predictions.

# true labels for each language, converted from strings to integers
true_labels = {
    lang: test_dfs[lang]['label'].map(LBL2ID).tolist()
    for lang in LANGUAGES
}


all_predictions = {} # storing predictions for every combo

for lang in LANGUAGES:
    all_predictions[lang] = {}
    for model_name, paths in MODEL_PATHS.items():
        print(f"Running inference: {model_name} on {lang}...")

         # load the model and get predictions for this language's test set
        preds = run_inference(paths[lang], test_dfs[lang])

        # store predictions under [language][model]
        all_predictions[lang][model_name] = preds
        print(f"  Done. {len(preds)} predictions made.\n")

print("All inference complete!")

Running inference: xlmr on hausa...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Done. 5303 predictions made.

Running inference: afriberta on hausa...


Loading weights:   0%|          | 0/169 [00:00<?, ?it/s]

  Done. 5303 predictions made.

Running inference: xlmr on kinyarwanda...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Done. 1026 predictions made.

Running inference: afriberta on kinyarwanda...


Loading weights:   0%|          | 0/169 [00:00<?, ?it/s]

  Done. 1026 predictions made.

All inference complete!


**5. Computing Metrics**

With predictions in hand, we now compare them against the true labels to compute F1, precision, recall and accuracy for each language + model combination. We also print a full per-class breakdown using sklearn's classification_report. This breaks the metrics down by sentiment class (negative, neutral and positive) so we can see where each model is strongest and weakest. That insight guides which disagreement cases we prioritise for LIME and SHAP analysis, specifically cases involving the classes each model struggled with most. The summary table at the end reports only the four metrics from our methodology.



In [ ]:
#conputing metrics


# store results in a list to build a comparison table at the end
results = []

for lang in LANGUAGES:
    true = true_labels[lang]
    for model_name in ['xlmr', 'afriberta']:
        preds = all_predictions[lang][model_name]

        # compute all four metrics using the true labels vs predictions
        f1        = f1_score(true, preds, average='weighted')
        precision = precision_score(true, preds, average='weighted', zero_division=0)
        recall    = recall_score(true, preds, average='weighted', zero_division=0)
        accuracy  = accuracy_score(true, preds)

        results.append({
            'language': lang,
            'model': model_name,
            'f1': round(f1, 4),
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'accuracy': round(accuracy, 4)
        })

        # also print a full per-class breakdown to tell us which sentiment class each model struggles with most
        print(f"\n{model_name.upper()} on {lang.upper()}")
        print(classification_report(true, preds, target_names=['negative', 'neutral', 'positive']))

# display everything as a clean summary table
results_df = pd.DataFrame(results)
print("\nSummary Table:")
display(results_df)


XLMR on HAUSA
              precision    recall  f1-score   support

    negative       0.69      0.74      0.71      1759
     neutral       0.71      0.68      0.69      1789
    positive       0.82      0.81      0.82      1755

    accuracy                           0.74      5303
   macro avg       0.74      0.74      0.74      5303
weighted avg       0.74      0.74      0.74      5303


AFRIBERTA on HAUSA
              precision    recall  f1-score   support

    negative       0.76      0.80      0.78      1759
     neutral       0.77      0.73      0.75      1789
    positive       0.86      0.87      0.86      1755

    accuracy                           0.80      5303
   macro avg       0.80      0.80      0.80      5303
weighted avg       0.80      0.80      0.80      5303


XLMR on KINYARWANDA
              precision    recall  f1-score   support

    negative       0.54      0.70      0.61       355
     neutral       0.59      0.34      0.43       393
    positive       

,language,model,f1,precision,recall,accuracy
0,hausa,xlmr,0.7406,0.7419,0.7403,0.7403
1,hausa,afriberta,0.7960,0.7963,0.7963,0.7963
2,kinyarwanda,xlmr,0.5266,0.5507,0.5400,0.5400
3,kinyarwanda,afriberta,0.6193,0.6192,0.6199,0.6199


**6. Disagreement extraction**

In this cell we identify every tweet where XLM-R and AfriBERTa produced different predictions. We combine the true labels and both models' predictions into a single dataframe per language, filter for the rows where the predictions differ, and save those to CSV files. These disagreement cases will be used for the LIME and SHAP analysis.

In [ ]:
disagreements = {}

for lang in LANGUAGES:
    xlmr_preds      = all_predictions[lang]['xlmr']
    afriberta_preds = all_predictions[lang]['afriberta']
    true            = true_labels[lang]

    # build a dataframe combining the tweet text, true label,
    # and both models' predictions side by side
    df = test_dfs[lang].copy()
    df['true_label']       = [ID2LBL[i] for i in true] #convert all integer vals back to original word
    df['xlmr_pred']        = [ID2LBL[i] for i in xlmr_preds]
    df['afriberta_pred']   = [ID2LBL[i] for i in afriberta_preds]


    disagreement_df = df[df['xlmr_pred'] != df['afriberta_pred']].copy() # keep only the rows in df where the two models disagree
    disagreements[lang] = disagreement_df

    print(f"{lang.upper()}: {len(disagreement_df)} disagreements out of {len(df)} tweets ({len(disagreement_df)/len(df)*100:.1f}%)")

    # save to CSV for XAI analysis
    out_path = os.path.join(PROJECT_PATH, f'outputs/metrics/disagreements_{lang}.csv')
    disagreement_df.to_csv(out_path, index=False)
    print(f"  Saved to {out_path}\n")

HAUSA: 1192 disagreements out of 5303 tweets (22.5%)
  Saved to /content/drive/Shareddrives/Cos760/outputs/metrics/disagreements_hausa.csv

KINYARWANDA: 400 disagreements out of 1026 tweets (39.0%)
  Saved to /content/drive/Shareddrives/Cos760/outputs/metrics/disagreements_kinyarwanda.csv

